# Basic Chat（单轮 / 多轮）

## 目标

本示例演示如何通过 Hy3 的 OpenAI 兼容 Chat Completions API 完成单轮对话，以及如何把上一轮 assistant 回复加入 `messages` 后继续多轮对话。

## 前置条件

- Python 3.10+
- OpenAI Python SDK `openai>=1.0.0`，环境变量加载库 `python-dotenv>=1.0.0`
- 环境变量：`HY3_BASE_URL`、`HY3_API_KEY`、`HY3_MODEL`；可从 `quickstart/.env.example` 复制为 `quickstart/.env`
- 模型能力要求：支持 Chat Completions 与多轮文本对话

安装依赖：

```bash
python -m pip install "openai>=1.0.0" "python-dotenv>=1.0.0"
```

## 完整请求

```python
messages = [
    {"role": "system", "content": "你是一个简洁、准确的中文助手。"},
    {"role": "user", "content": "请用一句话介绍 Hy3。"},
]

first_response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    temperature=0.9,
    top_p=1.0,
    max_tokens=128,
    extra_body={
        "chat_template_kwargs": {"reasoning_effort": "no_think"}
    },
)

first_answer = first_response.choices[0].message.content or ""
messages.extend(
    [
        {"role": "assistant", "content": first_answer},
        {"role": "user", "content": "请把刚才的介绍改写成三个要点。"},
    ]
)
second_response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    temperature=0.9,
    top_p=1.0,
    max_tokens=128,
    extra_body={
        "chat_template_kwargs": {"reasoning_effort": "no_think"}
    },
)
```

## 完整 Response 解析

脚本会遍历全部 `choices`，解析响应 ID、模型、创建时间、角色、正文、结束原因和 Token 用量。多轮对话必须把第一轮的 assistant 正文原样加入下一次请求，而不是只发送第二条用户消息。

```python
print(response.id, response.model, response.created)
for choice in response.choices:
    print(choice.index)
    print(choice.message.role)
    print(choice.message.content)
    print(choice.finish_reason)

if response.usage:
    print(response.usage.prompt_tokens)
    print(response.usage.completion_tokens)
    print(response.usage.total_tokens)
```

## 运行方式

在 `quickstart/` 目录执行：

```bash
python examples/01_basic_chat/basic_chat.py
```

## 示例输出

以下内容仅展示输出结构，回答文本、ID 和 Token 数会随部署与生成结果变化。

```text
=== 单轮对话 ===
id=chatcmpl-example-1
model=hy3
choice[0].role=assistant
choice[0].finish_reason=stop
choice[0].content=Hy3 是腾讯混元团队研发的混合专家大语言模型。
usage: prompt=25, completion=18, total=43

=== 多轮对话（第 2 轮） ===
id=chatcmpl-example-2
model=hy3
choice[0].role=assistant
choice[0].finish_reason=stop
choice[0].content=1. 混合专家架构。\n2. 支持长上下文。\n3. 支持推理与工具调用。
usage: prompt=58, completion=35, total=93
```


In [ ]:
"""Hy3 单轮与多轮基础对话示例。"""

import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI


def load_project_env() -> None:
    """从 quickstart/.env 或当前目录加载环境变量。"""
    candidates = [Path.cwd() / ".env", Path.cwd() / "quickstart" / ".env"]
    if "__file__" in globals():
        candidates.insert(0, Path(__file__).resolve().parents[2] / ".env")
    for candidate in candidates:
        if candidate.is_file():
            load_dotenv(candidate, override=False)
            return


load_project_env()

BASE_URL = os.getenv("HY3_BASE_URL", "http://127.0.0.1:8000/v1")
API_KEY = os.getenv("HY3_API_KEY", "EMPTY")
MODEL = os.getenv("HY3_MODEL", "hy3")
TEMPERATURE = float(os.getenv("HY3_TEMPERATURE", "0.9"))
TOP_P = float(os.getenv("HY3_TOP_P", "1.0"))
MAX_TOKENS = int(os.getenv("HY3_MAX_TOKENS", "128"))
REASONING_EFFORT = os.getenv("HY3_REASONING_EFFORT", "no_think")


def create_completion(client: OpenAI, messages: list[dict[str, str]]):
    """发送一次完整的非流式 Chat Completions 请求。"""
    return client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        max_tokens=MAX_TOKENS,
        extra_body={
            "chat_template_kwargs": {
                "reasoning_effort": REASONING_EFFORT,
            }
        },
    )


def parse_response(label: str, response) -> str:
    """解析响应元数据、全部 choices 和 token usage。"""
    print(f"\n=== {label} ===")
    print(f"id={response.id}")
    print(f"model={response.model}")
    print(f"created={response.created}")

    for choice in response.choices:
        message = choice.message
        print(f"choice[{choice.index}].role={message.role}")
        print(f"choice[{choice.index}].finish_reason={choice.finish_reason}")
        print(f"choice[{choice.index}].content={message.content}")
        if message.tool_calls:
            print(f"choice[{choice.index}].tool_calls={message.tool_calls}")

    if response.usage:
        print(
            "usage: "
            f"prompt={response.usage.prompt_tokens}, "
            f"completion={response.usage.completion_tokens}, "
            f"total={response.usage.total_tokens}"
        )

    return response.choices[0].message.content or ""


def main() -> None:
    client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

    # 单轮对话：一条用户消息对应一次回复。
    messages = [
        {"role": "system", "content": "你是一个简洁、准确的中文助手。"},
        {"role": "user", "content": "请用一句话介绍 Hy3。"},
    ]
    first_response = create_completion(client, messages)
    first_answer = parse_response("单轮对话", first_response)

    # 多轮对话：把上一轮 assistant 回复与新的 user 消息一起传回。
    messages.extend(
        [
            {"role": "assistant", "content": first_answer},
            {"role": "user", "content": "请把刚才的介绍改写成三个要点。"},
        ]
    )
    second_response = create_completion(client, messages)
    parse_response("多轮对话（第 2 轮）", second_response)


if __name__ == "__main__":
    main()
